# FAISS Vector Database for RAG

FAISS (Facebook AI Similarity Search) is a library that allows fast similarity search over dense vectors.

Instead of comparing a query with every vector, FAISS builds an index for fast retrieval.

In [3]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


In [4]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-l6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
chunks = [
    "Python is a popular programming language.",
    "Machine learning is a branch of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "RAG combines retrieval and generation.",
    "OpenCV is used for computer vision applications."
]

In [6]:
chunk_embeddings = model.encode(chunks)
print(chunk_embeddings.shape)

(5, 384)


In [7]:
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

In [8]:
dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(chunk_embeddings)

print('Total vectors in index:',index.ntotal)

Total vectors in index: 5


In [9]:
query = "What is deep learning?"

query_embedding = model.encode([query])

query_embedding = np.array(query_embedding).astype("float32")

In [10]:
k = 2

distances, indices = index.search(query_embedding, k)

print("Indices:", indices)
print("Distances:", distances)

Indices: [[2 1]]
Distances: [[0.60546386 0.88703525]]


In [11]:
for i, idx in enumerate(indices[0]):

    print(f"\nRank {i+1}")
    print("Chunk:", chunks[idx])
    print("Distance:", distances[0][i])


Rank 1
Chunk: Deep learning uses neural networks with many layers.
Distance: 0.60546386

Rank 2
Chunk: Machine learning is a branch of artificial intelligence.
Distance: 0.88703525


## Note

FAISS uses L2 distance by default.

If you want cosine similarity:

- Normalize embeddings
- Then use IndexFlatIP (Inner Product)

## Why FAISS is Used in Real RAG Systems

Without FAISS:
- Compare query with all chunks → slow

With FAISS:
- Pre-built index
- Sub-linear search
- Scales to millions of vectors

Used in:
- ChatGPT-like systems
- Semantic search engines
- Recommendation systems

- Document
-    ↓
- Chunking
-    ↓
- Embeddings
-    ↓
- FAISS Index
-    ↓
- Query Embedding
-    ↓
- Fast Retrieval
-    ↓
- Top-K Chunks
-    ↓
- LLM